## **Data Level Security Using ABAC (Governed Tags)**

This notebook demonstrates how data-level security can be implemented using Attribute-Based Access Control (ABAC) in Unity Catalog.

In previous examples, row filters and column masks were applied direcrlt to indivdual tables. While effective, the approach requires policies to be defined and managed seperately for each table.

ABAC addresses this by introducing a centralized model where:

- Data is tagged using governed attributes
- Policies are defined once based on those attributes
- Policies are applied automatically across multiple objects

The objective is to:

- Classify sensitive data using governed tags
- Define reusable policies based on those tags
Note: Row filters and column masks are not supported on Dedicated (single User) clusters as of now. Use a Shared cluster or a Serverless cluster to run this demo.

**Step 1: Set up the schema for the table**

In [0]:
USE CATALOG demo;
USE SCHEMA data_security;

**## Step 2: Create the table**

A table is defined to represent sales data across multiple regions.

In [0]:
CREATE OR REPLACE TABLE sales_abac (
    id INT,
    region STRING,
    email STRING,
    revenue INT
);

**Step 3: Insert Sample Data**

In [0]:
INSERT INTO sales_abac VALUES
(1, 'UK', 'john.doe@email.com', 10000),
(2, 'US', 'mike.smith@email.com', 20000),
(3, 'UK', 'sara.jones@email.com', 15000),
(4, 'US', 'david.brown@email.com', 25000),
(5, 'UK', 'emma.wilson@email.com', 18000);

In [0]:
SELECT * FROM sales_abac;

**Step 4: Create UDFs for row filter and column mask**

(We have created them as part of previous lecture)

1. fn_filter_region (This function evaluates the user's group membership and determines whether to include specific record in the output based on the region)
2. fn_mask_email (This function evaluates the user's group memberhsip and determines whether to return the original value or masked version of the value)

**Step 5: Create Governed Tags**

1. pii = email (indicates that the data contains personally identifiable information (PII))
2. access_type = region (Defines the type of access control to applied to the data, such as region-based  or department-based filtering)

Step 6: Create Policy

(Could be at catalog schema/ table level)

    1. policy_filter_region
    2. policy_mask_email

Step 7: Attach tags to the table

1. sales_abac_region
2. sales_abac_email

In [0]:
SELECT * FROM sales_abac;

**Step 8: Create another table to demonstrate scalability of the ABAC implementation**

In [0]:
CREATE OR REPLACE TABLE customer_abac (
    customer_id INT,
    customer_name STRING,
    email STRING,
    region STRING
);

In [0]:
ALTER TABLE customer_abac
ALTER COLUMN email
SET TAGS ('pii' = 'email');

In [0]:
ALTER TABLE customer_abac
ALTER COLUMN region
SET TAGS ('access_type' = 'region');

In [0]:
INSERT INTO customer_abac VALUES
(1, 'John Smith', 'john.smith@email.com', 'UK'),
(2, 'Mike Johnson', 'mike.johnson@email.com', 'US'),
(3, 'Sara Jones', 'sara.jones@email.com', 'UK'),
(4, 'David Brown', 'david.brown@email.com', 'US');

In [0]:
select * from customer_abac;